# WildFL — Federated Learning for Wildlife Detection

This notebook demonstrates how to run the federated-learning (FL) algorithms implemented in `algorithms/`:
**FedAvg**, **FedBN**, **FedNova**, **MOON**, **FedDC**, and **FedProto**.

The algorithms federate [Ultralytics](https://docs.ultralytics.com/) YOLO detectors. A customized fork of the `ultralytics` framework is required to execute th below code, since in every round, local training must start from the aggregated server weights. The algorithms therefore pass an additional `state=<path to a state dict>` argument to `model.train()`, which stock Ultralytics does not support. MOON and FedProto further rely on the fork's model-contrastive andprototype-alignment loss terms.

In [ ]:
import sys
import json

from pathlib import Path

# make the algorithm classes importable (the notebook runs from the repository root)
sys.path.append("algorithms")

from helpers import init_rng, NAMES_DIRECTORIES_FILES
from FedAvg import FedAvg
from FedBN import FedBN
from FedNova import FedNova
from MOON import MOON
from FedDC import FedDC
from FedProto import FedProto

init_rng(0)  # seed the Python, NumPy and PyTorch RNGs for reproducibility

## 1. Shared configuration

All experiments below share one configuration dictionary. The keys mirror the arguments of the algorithms'
`__call__` methods:

| Key | Meaning |
|---|---|
| `rounds` | number of communication rounds |
| `mdl_type`, `mdl_version` | Ultralytics model family and size; the model is built from `<mdl_type><mdl_version>.yaml` (e.g. `yolo11n.yaml`) |
| `clients` | names of the client subdirectories inside `data_dir` |
| `data_dir` | root of the prepared dataset |
| `cfg_dir` | directory of the Ultralytics model-configuration YAMLs, i.e. the folder containing `<mdl_type>.yaml` |
| `class_id2name` | mapping of integer class ids to class names |
| `ann_format` | annotation format of `annotations.json`; `"BX_WH"` is the correct chocie for bounding box detection and denotes `[x_min, y_min, width, height]` boxes.  |
| `training_params` | forwarded unchanged to Ultralytics' `model.train()` for every client |
| `img_format` | image file extension |
| `eval_split` | subdirectory of each client used for evaluation |
| `use_tiling` | if `True`, evaluation images are processed patch-wise with SAHI tiled inference |
| `eval_global` | if `True`, the aggregated global model is additionally evaluated on every client after each round |
| `patch_dims`, `patch_overlap` | tile size and overlap used for tiled inference |

<br/>
<br/>  


> **The parameter `val` within the `training_params` dict must be always be `False`:** with validation enabled, Ultralytics restores the *best* checkpoint at the end of local training — but federated aggregation must combine the *last* local weights. Every algorithm asserts `training_params["val"] == False` for this reason; `patience` is set high to disable early stopping as well.

In [ ]:
config = {
    "rounds":        30,
    "mdl_type":      "yolo11",
    "mdl_version":   "n",
    "clients":       ["c1", "c2", "c3", "c4", "c5"],
    "data_dir":      "/path/to/prepared_dataset",           
    "cfg_dir":       "/path/to/ultralytics/cfg/models/11",  # <-- adapt: folder containing yolo11.yaml
    "class_id2name": {0: "Elephant"},
    "ann_format":    "BX_WH",
    "training_params": {
        "epochs":   20,      # local epochs per round
        "batch":    32,
        "imgsz":    640,
        "device":   0,
        "patience": 1000,    # effectively disables early stopping
        "amp":      False,
        "val":      False,   # must be False (see the note above)
    },
    "img_format":    "jpg",
    "eval_split":    "val_full_imgs",
    "use_tiling":    True,
    "eval_global":   True,
    "patch_dims":    {"width": 640, "height": 640},
    "patch_overlap": 0.2,
}


def make_output_dir(experiment_name: str) -> str:
    """Create an experiment directory and store the configuration alongside the results."""
    out = Path("experiments") / experiment_name
    out.mkdir(parents=True, exist_ok=True)
    with open(out / "config.json", "w") as f:
        json.dump(config, f, indent=1)
    return str(out)

## 2. Running the algorithms

All six algorithms share the same two-step interface:

```python
algo = Algorithm(<model / schedule configuration>)   # 1. construct
results = algo(<data + training configuration>)      # 2. run the federation
```

Every communication round proceeds in three steps:

1. **Local training** — each participating client updates a model on its local training patches, starting from the
   current server state.
2. **Aggregation** — the server combines the client updates; the weight of each client is proportional to its
   number of training annotations (read from `client_stats.json`).
3. **Evaluation** — every local model is evaluated on its own client's `eval_split`; with `eval_global=True`, the
   aggregated global model is additionally evaluated on all clients.

The per-round Ultralytics training runs are written to `<output_dir>/round<i>/<client>/` (pass
`rm_client_data=True` to delete them after every round and save disk space). Metrics are streamed to
`<output_dir>/result_<algorithm>.json` (global model) and `<output_dir>/result_<algorithm>_local.json`
(local models) after every round, so intermediate results survive interruptions.

Below a brief introduction into the different algorithms is given. For more detail, please refer to "XY".

### 2.1 FedAvg
After each round of local training, the server replaces the global model with the *weighted average* of the clients' parameters, and the averaged model is redistributed as the starting point of the next round. In this implementation, client weights are computed from the number of local training annotations (rather than the number of images)

In [ ]:
fedavg = FedAvg(rounds=config["rounds"], mdl_type=config["mdl_type"], mdl_version=config["mdl_version"])

output_dir = make_output_dir("FedAvg_20E_30R")

results = fedavg(clients=config["clients"], data_dir=config["data_dir"], cfg_dir=config["cfg_dir"],
                 class_id2name=config["class_id2name"], ann_format=config["ann_format"],
                 training_params=config["training_params"], img_format=config["img_format"],
                 eval_split=config["eval_split"], use_tiling=config["use_tiling"], output_dir=output_dir,
                 patch_dims=config["patch_dims"], patch_overlap=config["patch_overlap"],
                 eval_global=config["eval_global"])

### 2.2 FedBN
All parameters are averaged exactly as in FedAvg, **except the BatchNorm layers, which stay local** to each client.

In [ ]:
fedbn = FedBN(rounds=config["rounds"], mdl_type=config["mdl_type"], mdl_version=config["mdl_version"])

output_dir = make_output_dir("FedBN_20E_30R")

results = fedbn(clients=config["clients"], data_dir=config["data_dir"], cfg_dir=config["cfg_dir"],
                class_id2name=config["class_id2name"], ann_format=config["ann_format"],
                training_params=config["training_params"], img_format=config["img_format"],
                eval_split=config["eval_split"], use_tiling=config["use_tiling"], output_dir=output_dir,
                patch_dims=config["patch_dims"], patch_overlap=config["patch_overlap"],
                eval_global=config["eval_global"])

### 2.3 FedNova

 FedNova normalizes each client's cumulative update by its number of local steps before averaging at the server (which is then done in the same way as FedAvg). Here, the number of steps is derived from each client's patch count in `client_stats.json` together with the batch size and local epochs.

In [ ]:
fednova = FedNova(rounds=config["rounds"], mdl_type=config["mdl_type"], mdl_version=config["mdl_version"])

output_dir = make_output_dir("FedNova_20E_30R")

results = fednova(clients=config["clients"], data_dir=config["data_dir"], cfg_dir=config["cfg_dir"],
                  class_id2name=config["class_id2name"], ann_format=config["ann_format"],
                  training_params=config["training_params"], img_format=config["img_format"],
                  eval_split=config["eval_split"], use_tiling=config["use_tiling"], output_dir=output_dir,
                  patch_dims=config["patch_dims"], patch_overlap=config["patch_overlap"],
                  eval_global=config["eval_global"])

### 2.4 MOON

The MOON algorithm  adds a contrastive term to the local training
objective: the representation of the current local model is pulled toward that of the global model and pushed away
from that of the previous local model, which limits client drift on heterogeneous data. Two extra hyperparameters
are added to `training_params`: `moon` (µ, the weight of the contrastive term) and `tau_moon` (the temperature of
the contrastive loss).

> **Implementation note.** Before each round, MOON pre-computes the feature embeddings that the contrastive loss
> compares against (those of the global and of the previous local model) for every training patch, and stores them
> as `.npy` files inside the client training directories. Use a dedicated copy of `data_dir` if the original should
> stay untouched.

In [ ]:
moon_training_params = dict(config["training_params"])
moon_training_params.update({"moon": 1.0,        # weight of the model-contrastive loss term
                             "tau_moon": 0.5})   # temperature of the contrastive loss

moon = MOON(rounds=config["rounds"], mdl_type=config["mdl_type"], mdl_version=config["mdl_version"])

output_dir = make_output_dir("MOON_1.0MU_0.5TAU_20E_30R")

results = moon(clients=config["clients"], data_dir=config["data_dir"], cfg_dir=config["cfg_dir"],
               class_id2name=config["class_id2name"], ann_format=config["ann_format"],
               training_params=moon_training_params, img_format=config["img_format"],
               eval_split=config["eval_split"], use_tiling=config["use_tiling"], output_dir=output_dir,
               patch_dims=config["patch_dims"], patch_overlap=config["patch_overlap"],
               eval_global=config["eval_global"])

### 2.5 FedDC

In FedDc, before every aggregation event, models are *passed around from client to client* in random order, so every model effectively sees more data than any single client holds. The constructor takes a different set of arguments:

- `agg_events` — number of aggregation events (one "round" in the result files corresponds to one aggregation);
- `dc_period` — number of local epochs a model trains on one client before being passed to the next;
- `agg_period` — number of epochs between two aggregations (must be a multiple of `dc_period`).

Because the training length is fully determined by the two periods, `epochs` **must not** appear in `training_params` for this algorithm.

In [ ]:
feddc_training_params = {k: v for k, v in config["training_params"].items() if k != "epochs"}

feddc = FedDC(agg_events=30, dc_period=4, agg_period=20,
              mdl_type=config["mdl_type"], mdl_version=config["mdl_version"])

output_dir = make_output_dir("FedDC_4DC_20AGG_30R")

results = feddc(clients=config["clients"], data_dir=config["data_dir"], cfg_dir=config["cfg_dir"],
                class_id2name=config["class_id2name"], ann_format=config["ann_format"],
                training_params=feddc_training_params, img_format=config["img_format"],
                eval_split=config["eval_split"], use_tiling=config["use_tiling"], output_dir=output_dir,
                patch_dims=config["patch_dims"], patch_overlap=config["patch_overlap"],
                eval_global=config["eval_global"])

### 2.6 FedProto

FedProto uses **class prototypes instead of model weights**: after local training, each client computes the mean feature embedding of its annotated objects per class, and the server aggregates these into global prototypes. From the second round on, an alignment term regularizes local training by pulling each client's object embeddings toward the global prototypes.

In [ ]:
fedproto = FedProto(rounds=config["rounds"], mdl_type=config["mdl_type"], mdl_version=config["mdl_version"])

output_dir = make_output_dir("FedProto_20E_30R")

results = fedproto(clients=config["clients"], data_dir=config["data_dir"], cfg_dir=config["cfg_dir"],
                   class_id2name=config["class_id2name"], ann_format=config["ann_format"],
                   training_params=config["training_params"], inf_params={},
                   img_format=config["img_format"], eval_split=config["eval_split"],
                   use_tiling=config["use_tiling"], output_dir=output_dir,
                   patch_dims=config["patch_dims"], patch_overlap=config["patch_overlap"])

## 3. Inspecting the results

All algorithms return — and continuously write — nested dictionaries of per-round metrics, evaluated per client:

```json
{
 "round0": {
  "c1": {
   "count_err": {"0": {"MAE": "...", "ME": "...", "MSE": "...", "MARE": "...", "MRE": "..."},
                 "binary": {"...": "..."}},
   "em":        {"0": {"confusion": "...", "precision": "...", "recall": "...", "f1": "...",
                       "metrics/precision(B)": "...", "metrics/recall(B)": "...",
                       "metrics/mAP50(B)": "...", "metrics/mAP50-95(B)": "..."},
                 "binary": {"...": "..."}}
  },
  "c2": "..."
 },
 "round1": "..."
}
```

Metrics are reported per class id (`"0"`, `"1"`, ...) and class-agnostic (`"binary"`), in two groups:

| Group | Metrics | Meaning |
|---|---|---|
| `count_err` | `MAE`, `ME`, `MSE`, `MARE`, `MRE` | image-level counting errors: predicted vs. ground-truth animal counts per image (mean absolute / signed / squared / relative errors) |
| `em` | `precision`, `recall`, `f1`, `confusion` | detection metrics aggregated from image-level confusion matrices (`confusion` is the per-class confusion rate, in %; see [this paper](https://www.sciencedirect.com/science/article/pii/S1574954125003966))|
| `em` | `metrics/precision(B)`, `metrics/recall(B)`, `metrics/mAP50(B)`, `metrics/mAP50-95(B)` | standard Ultralytics box-detection metrics |

Two groups of evaluation metrics (`em`) are computed: the plain-name metrics (e.g., `precision`) in the second row of the above table are derived from confusion matrices, calculated at a fixed confidence threshold. The metrics in the last row  (e.g., `metrics/precision(B)`), on the other hand, are sampled from the precision-recall curve generated by evaluation pipeline built into the `ultralytics` framework (cf. [the official documentation for more detail](https://docs.ultralytics.com/guides/yolo-performance-metrics#introduction)). In our paper "XY", only the latter group was used. 
